# Tái lập run Vast `only_consformer` (test EM 31.72) trên Colab

Run cũ `Kus669/ViSceT5-ft-vitextvqa-only_consformer` được train trên Vast ngày 08/08 bằng commit **`7b99d60`** (nhánh `exp/pretrain-gen-all`; run chạy 09:38–15:18 UTC, sau commit này và trước commit kế tiếp). Cấu hình: CLIP không câu hỏi + ConsFormer, không OCR-Aug, không Visual Search; batch 4×2, chọn best theo loss.

Notebook chạy **đúng code của commit đó** (clone riêng vào `/content/ViSceT5_old`, detached HEAD, không sửa dòng nào) bằng `scripts/repro_old_commit.sh` → `setup.sh` → `run_pipeline.py` của commit cũ. Repo hiện tại (`/content/ViSceT5`, nhánh `exp/ablation-drop-ocr`) chỉ cung cấp công cụ: tải checkpoint (`hf_io.py`), chấm điểm (`benchmark_eval.py`), so sánh (`repro_compare.py`), ghi môi trường (`env_report.py`) và file nộp gốc của Vast (`repro/vast_only_consformer_test.csv`).

| Bước | Làm gì | Trả lời câu hỏi |
|---|---|---|
| **1. Predict lại** | Lấy checkpoint gốc của Vast (`checkpoint-21975`, `checkpoint-13185`), predict test bằng code cũ trên Colab | Môi trường Colab có làm **inference** khác đi không? Checkpoint nào sinh ra 31.72? |
| **2. Train lại** | Train lại từ đầu bằng commit cũ, cùng seed, cùng cấu hình, rồi predict + chấm | Môi trường/phần cứng có làm **huấn luyện** cho kết quả khác không? |

**Đọc kết quả**

| Bước 1 | Bước 2 | Kết luận |
|---|---|---|
| ≈31.72, trùng ~100% với file Vast | ≈31.7 | Code mới gây chênh lệch → so code cũ/mới |
| ≈31.72 | ≈30.9 hoặc lệch nhiều so với 31.72 | Phần cứng/môi trường + dao động khi train (không phải code) |
| Khác 31.72 rõ rệt | — | Môi trường đổi cả kết quả inference (torch/CUDA/GPU) |

Bước 2 mất khoảng 5–6 giờ trên A100. Checkpoint được push lên HF mỗi epoch; Colab ngắt thì chạy lại cell 1–3 rồi cell train, code cũ tự resume từ HF.

## 1. Lấy repo công cụ (nhánh `exp/ablation-drop-ocr`)

In [ ]:
import os
BRANCH = "exp/ablation-drop-ocr"
if not os.path.exists("/content/ViSceT5"):
    !git clone -b {BRANCH} https://github.com/Kussssssss/ViSceT5.git /content/ViSceT5
%cd /content/ViSceT5
!git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
!git log --oneline -3
for f in ("scripts/repro_old_commit.sh", "scripts/repro_compare.py", "scripts/env_report.py",
          "repro/vast_only_consformer_test.csv", "eval_gt/ViTextVQA_test_gt.json"):
    assert os.path.exists(f), f"Thiếu {f} — kiểm tra lại nhánh!"

## 2. Hugging Face token
Cần để tải checkpoint gốc, push checkpoint của lần train lại (resume khi Colab ngắt) và lưu kết quả. Thêm secret `HF_TOKEN` (tab 🔑, bật *Notebook access*) hoặc chạy cell rồi dán token.

In [ ]:
import os
try:
    from google.colab import userdata
    _t = userdata.get("HF_TOKEN")
    if _t:
        os.environ["HF_TOKEN"] = _t
except Exception as e:
    print("Colab Secrets:", e)
if not os.environ.get("HF_TOKEN"):
    from huggingface_hub import notebook_login
    notebook_login()
else:
    print("✅ Đã có HF_TOKEN từ Colab Secrets.")

## 3. Cấu hình
`REPRO_REPO` là repo HF mới chứa checkpoint của lần train lại và mọi kết quả của notebook này (`step1/…`, `step2/…`, `reports/…`). Đổi `REPRO_TAG` nếu muốn chạy lại thêm một seed/lần nữa vào repo khác.

In [ ]:
OLD_COMMIT = "7b99d60"
OLD_REPO   = "Kus669/ViSceT5-ft-vitextvqa-only_consformer"
OLD_CKPTS  = ["checkpoint-21975", "checkpoint-13185"]
REPRO_TAG  = "c1"

import os
from huggingface_hub import get_token, HfApi
_tok = os.environ.get("HF_TOKEN") or get_token()
if _tok:
    os.environ["HF_TOKEN"] = _tok
HF_USER = "Kus669"
if _tok:
    try:
        HF_USER = HfApi(token=_tok).whoami().get("name") or HF_USER
    except Exception as e:
        print("whoami lỗi:", e)
else:
    print("⚠️ CHƯA có HF_TOKEN: không tải được checkpoint gốc nếu repo private, và checkpoint train lại sẽ mất khi ngắt kết nối!")

REPRO_REPO = f"{HF_USER}/ViSceT5-repro-{OLD_COMMIT}-only_consformer-{REPRO_TAG}"
OLD_DIR    = "/content/ViSceT5_old"
WORK       = "/content/repro"
GT         = "/content/ViSceT5/eval_gt/ViTextVQA_test_gt.json"
VAST_TEST  = "/content/ViSceT5/repro/vast_only_consformer_test.csv"
os.makedirs(WORK, exist_ok=True)

os.environ.update({
    "REPRO_COMMIT":         OLD_COMMIT,
    "REPRO_DIR":            OLD_DIR,
    "ABLATION_USE_QACLIP":  "false",
    "ABLATION_USE_VS":      "false",
    "ABLATION_USE_OCR":     "true",
    "ABLATION_USE_OCR_AUG": "false",
})
print(f"Commit cũ  : {OLD_COMMIT} → {OLD_DIR}")
print(f"Run gốc    : {OLD_REPO} {OLD_CKPTS}")
print(f"REPRO_REPO : {REPRO_REPO}")
print("Cấu hình train lại: qaclip=false | vs=false | ocr(ConsFormer)=true | ocr_aug=false; còn lại theo configs/finetune.yaml của commit cũ")

## 4. Hàm dùng chung: predict bằng code cũ, chấm điểm, upload
`predict_old(model_dir, pred_dir, name)` gọi `STAGE=predict` của commit cũ trên checkpoint `model_dir`, chép `submission_{dev,test}.csv` + log vào `pred_dir`, chấm test bằng `benchmark_eval.py` (đúng giao thức trong bài) và upload lên `REPRO_REPO/<name>/`. Đã có `submission_test.csv` trong `pred_dir` thì bỏ qua predict.

In [ ]:
import os, json, shutil

def predict_old(model_dir, pred_dir, name):
    os.makedirs(pred_dir, exist_ok=True)
    assert os.path.exists(os.path.join(model_dir, "model.safetensors")), f"Không có model trong {model_dir}"
    test_csv = os.path.join(pred_dir, "submission_test.csv")
    if os.path.exists(test_csv):
        print(f"✅ Đã có {test_csv} → bỏ qua predict.")
    else:
        for sp in ("dev", "test"):
            p = f"{OLD_DIR}/output/submission_{sp}.csv"
            if os.path.exists(p):
                os.remove(p)
        log = os.path.join(pred_dir, "predict_log.txt")
        !STAGE=predict PREDICT_SPLIT=both PREDICT_CKPT_DIR={model_dir} HF_REPO= REPRO_LOG={log} bash scripts/repro_old_commit.sh predict
        for sp in ("dev", "test"):
            src = f"{OLD_DIR}/output/submission_{sp}.csv"
            if os.path.exists(src):
                shutil.copy2(src, os.path.join(pred_dir, f"submission_{sp}.csv"))
    log = os.path.join(pred_dir, "predict_log.txt")
    if os.path.exists(log):
        dev = [l.strip() for l in open(log, encoding="utf-8", errors="ignore") if "[dev] EM=" in l]
        if dev:
            print(f"[{name}] {dev[-1]}")
    if os.path.exists(test_csv):
        !python -m training.benchmark_eval --gt {GT} --pred {test_csv} --out {pred_dir}/eval_test.json
    if os.environ.get("HF_TOKEN"):
        !python scripts/hf_io.py upload_predictions --repo {REPRO_REPO} --dir {pred_dir} --path_in_repo {name} --files submission_dev.csv submission_test.csv predict_log.txt eval_test.json
    return test_csv

!pip install -q evaluate bert-score git+https://github.com/salaniz/pycocoevalcap

## 5. Bước 1 — predict lại checkpoint gốc của Vast bằng code cũ trên Colab
Chỉ tải file inference của từng checkpoint (không tải optimizer). Sau khi xong, bảng so sánh cho biết EM/F1 và **tỷ lệ dự đoán trùng file nộp gốc của Vast**: trùng ~100% nghĩa là inference trên Colab cho ra đúng như Vast, và checkpoint đó chính là checkpoint đã sinh ra 31.72.

In [ ]:
import json
STEP1 = {}
for ck in OLD_CKPTS:
    info_path = f"{WORK}/info_step1_{ck}.json"
    if os.path.exists(info_path):
        os.remove(info_path)
    !python scripts/hf_io.py prepare --repo {OLD_REPO} --dir {WORK}/step1 --ckpt {ck} --info_out {info_path}
    info = json.load(open(info_path, encoding="utf-8"))
    STEP1[ck] = predict_old(info["model_dir"], f"{WORK}/step1/pred_{ck}", f"step1/{ck}")

!python scripts/env_report.py --repo_dir {OLD_DIR} --out {WORK}/env_colab.json
preds = " ".join([f"vast_original={VAST_TEST}"] + [f"colab_predict_{ck}={p}" for ck, p in STEP1.items()])
!python scripts/repro_compare.py --gt {GT} --ref {VAST_TEST} --pred {preds} --out {WORK}/step1_compare.json
if os.environ.get("HF_TOKEN"):
    !python scripts/hf_io.py upload_predictions --repo {REPRO_REPO} --dir {WORK} --path_in_repo reports --files env_colab.json step1_compare.json

## 6. Bước 2a — (tuỳ chọn) mock test commit cũ
Vài bước `--smoke_test` để chắc commit cũ chạy được với Colab hiện tại trước khi train 5–6 giờ. Không push, không resume; thư mục output của mock bị xoá sau đó.

In [ ]:
!MOCK_TEST=true HF_REPO= REPRO_LOG={WORK}/mock_log.txt bash scripts/repro_old_commit.sh finetune mock
!rm -rf {OLD_DIR}/output/finetune

## 7. Bước 2b — train lại từ đầu bằng commit cũ
Mọi thông số lấy từ `configs/finetune.yaml` của commit `7b99d60` (5 epoch, batch 4×2, lr 3e-5, cosine, warmup 0.1, wd 0.1, seed 42, best theo loss). Mỗi checkpoint được push lên `REPRO_REPO`; Colab ngắt thì chạy lại cell 1–4 rồi cell này để resume. Log nối vào `/content/repro/train_log.txt`.

In [ ]:
!HF_REPO={REPRO_REPO} REPRO_LOG={WORK}/train_log.txt bash scripts/repro_old_commit.sh finetune
!python scripts/env_report.py --repo_dir {OLD_DIR} --out {WORK}/env_colab_train.json

## 8. Bước 2c — predict + chấm model train lại, so với Vast
Như run gốc: `best` = model ở thư mục gốc (best theo loss, do `load_best_model_at_end`), `latest` = checkpoint epoch 5. Model lấy từ `/content/ViSceT5_old/output/finetune` nếu còn, không thì tải từ `REPRO_REPO`. Bảng cuối gộp cả bước 1 và bước 2; cột `same_as_ref_%` và `dEM_95CI` đo so với file nộp gốc của Vast.

In [ ]:
import json
STEP2 = {}
for ck in ("best", "latest"):
    info_path = f"{WORK}/info_step2_{ck}.json"
    if os.path.exists(info_path):
        os.remove(info_path)
    !python scripts/hf_io.py prepare --repo {REPRO_REPO} --dir {OLD_DIR}/output/finetune --ckpt {ck} --info_out {info_path}
    info = json.load(open(info_path, encoding="utf-8"))
    STEP2[ck] = predict_old(info["model_dir"], f"{WORK}/step2/pred_{info['ckpt']}", f"step2/{info['ckpt']}")

STEP1 = {ck: f"{WORK}/step1/pred_{ck}/submission_test.csv" for ck in OLD_CKPTS}
preds = " ".join([f"vast_original={VAST_TEST}"]
                 + [f"colab_predict_{ck}={p}" for ck, p in STEP1.items()]
                 + [f"colab_retrain_{ck}={p}" for ck, p in STEP2.items()])
!python scripts/repro_compare.py --gt {GT} --ref {VAST_TEST} --pred {preds} --out {WORK}/step2_compare.json
if os.environ.get("HF_TOKEN"):
    !python scripts/hf_io.py upload_predictions --repo {REPRO_REPO} --dir {WORK} --path_in_repo reports --files env_colab_train.json step2_compare.json train_log.txt

## 9. Đường dev EM theo epoch: Vast gốc so với lần train lại

In [ ]:
import json, glob
!python -c "from huggingface_hub import hf_hub_download as d; print(d('{OLD_REPO}', '{OLD_CKPTS[0]}/trainer_state.json', local_dir='{WORK}/vast_state'))"
def curve(path):
    st = json.load(open(path, encoding="utf-8"))
    return {round(h["epoch"]): (round(100 * h["eval_em"], 2), round(100 * h["eval_f1"], 2), round(h["eval_loss"], 4))
            for h in st["log_history"] if "eval_em" in h}
old_state = sorted(glob.glob(f"{WORK}/vast_state/checkpoint-*/trainer_state.json"))
new_state = sorted(glob.glob(f"{OLD_DIR}/output/finetune/checkpoint-*/trainer_state.json"),
                   key=lambda p: int(p.split("checkpoint-")[-1].split("/")[0]))
old_c = curve(old_state[-1]) if old_state else {}
new_c = curve(new_state[-1]) if new_state else {}
print(f"{'epoch':>5} | {'Vast EM':>8} {'F1':>6} {'loss':>7} | {'Colab EM':>8} {'F1':>6} {'loss':>7}")
for ep in sorted(set(old_c) | set(new_c)):
    o, n = old_c.get(ep, ("-",) * 3), new_c.get(ep, ("-",) * 3)
    print(f"{ep:>5} | {o[0]:>8} {o[1]:>6} {o[2]:>7} | {n[0]:>8} {n[1]:>6} {n[2]:>7}")